# 01 — Java Essentials

**What you'll learn:**

- What Java is and why it still anchors enterprise systems in 2026
- How the JVM compile-and-run pipeline works
- How to install Java 21 with Eclipse Temurin
- Three ways to run Java code: `jshell`, `jbang`, and Maven
- Values, primitive vs reference types, and autoboxing
- `var` and local-variable type inference
- Expressions vs statements — and where `switch` blurs the line
- Operators (and the `==` vs `.equals()` gotcha)
- Control flow: `if`, `while`, `for`, `switch`
- Methods, overloading, and `void`
- Strings, text blocks, and the immutability rule
- Comments and Javadoc

Prior programming experience helps but is not required. If you have written Python or Scala, you will see familiar shapes in new clothing — Java is more verbose than both, and the trade you get for that verbosity is a type system you can read at a glance and a runtime hardened by twenty-five years of production use.

## What is Java?

Java is a **statically typed, class-based, object-oriented** language that compiles to bytecode and runs on the **Java Virtual Machine** (JVM).

It was created by James Gosling and his team at Sun Microsystems in 1995 with one famous promise: *write once, run anywhere*. The compiler does not produce machine code for your CPU — it produces JVM bytecode, and any JVM on any operating system can load and execute it.

Four traits that define modern Java in one breath:

- **Statically typed** — every variable has a type the compiler knows at compile time, so type mismatches are caught before the program runs.
- **Class-based OOP** — code lives inside classes; classes are the unit of encapsulation, inheritance, and the file system layout.
- **Memory-managed** — the JVM's garbage collector reclaims unreachable objects for you; you do not call `free`.
- **Backwards compatible to a fault** — code written in 1998 still compiles and runs on Java 21 with very few exceptions. The ecosystem rewards stability.

## Why Java in 2026?

Three reasons people pick Java today:

| Reason | What it unlocks |
|---|---|
| **Enterprise ubiquity** | Banks, telecoms, governments, e-commerce platforms — the largest backends in the world are Java. The hiring market is enormous. |
| **The Spring ecosystem** | Spring and Spring Boot are the dominant backend framework on the JVM. Most production microservices you encounter are Spring services. |
| **JVM performance + tooling** | A mature garbage collector, a just-in-time compiler that optimises hot code paths, and the best profiling tools on any platform. |

For this learning track the **Spring connection** is the centre of gravity. Notebooks 01 through 07 build the Java foundation; notebooks 08 through 12 turn that foundation into real Spring services.

## Modern Java, not 1998 Java

This track targets **Java 21**, the current long-term-support release. Java has evolved enormously since the early days, and a lot of older tutorials still teach the 1998 dialect. We will write modern Java throughout.

Five features you will use constantly that did not exist in old Java:

- **`var`** — local-variable type inference (Java 10). You write `var x = ...` and the compiler infers the type.
- **Text blocks** — multi-line string literals with `"""` (Java 15).
- **Records** — one-line immutable data classes (Java 16). You will meet these in notebook 02.
- **Pattern matching for `switch`** — `switch` as an expression that destructures values (Java 21).
- **Virtual threads** — millions of cheap threads on the same JVM (Java 21). You will meet these in notebook 06.

If you stumble onto a Stack Overflow answer that defines a class just to hold three fields with getters and setters, that's old Java. The shape of the problem is the same; modern Java solves it in one line with a `record`.

## The JVM context

When you write Java, the compiler `javac` turns your source code into **JVM bytecode** — a portable instruction set. The JVM then loads that bytecode and executes it.

```text
   Main.java
       |
       v
    [ javac ]         <- Java compiler
       |
       v
   Main.class         <- JVM bytecode (portable)
       |
       v
     [ JVM ]          <- loads bytecode, GC, JIT-compiles hot paths
       |
       v
     output
```

Two consequences worth absorbing now:

1. **Startup is not instant.** The JVM itself takes a moment to boot. For tiny scripts this feels slow; for long-running services it disappears into the noise.
2. **You inherit the JVM's strengths.** A mature garbage collector, a JIT compiler that gets *faster* the longer your program runs, and decades of profiling tools (`jstack`, `jmap`, `jfr`, `async-profiler`).

## Installing Java

The recommended distribution is **Eclipse Temurin** — a free, production-grade build of OpenJDK maintained by the Adoptium project. On macOS, install it with Homebrew:

In [ ]:
%%bash
brew install --cask temurin     # JDK 21 LTS
brew install jbang              # single-file Java script runner
brew install maven gradle       # multi-file project build tools

Verify the install in a new terminal so the updated `PATH` takes effect:

In [ ]:
%%bash
java --version       # expect: openjdk 21.x.x
javac --version      # the compiler
jshell --version     # the REPL (ships with the JDK)
jbang --version      # single-file runner
mvn --version        # Maven build tool

## Three ways to run Java

Java gives you three on-ramps. Pick the one that matches the size of what you are doing.

```text
  +-------------+   +---------------+   +------------------+
  |   jshell    |   |     jbang     |   |   Maven/Gradle   |
  |  (one-off)  |   |  (one file)   |   |  (real project)  |
  +-------------+   +---------------+   +------------------+
   live eval        run a .java       many files, deps,
   no file          like a script     tests, build config
```

- **`jshell`** — for *what does this expression do?* The cheapest feedback loop. Ships with the JDK.
- **`jbang`** — for a single file you want to keep around. Add dependencies via a comment at the top.
- **Maven / Gradle** — for anything that has more than one file or that you want to build, test, and deploy seriously. This is what every Spring project uses.

## Hello, jshell

Open the REPL with the `jshell` command. You get a `jshell>` prompt that evaluates one expression or statement at a time.

In [ ]:
jshell> System.out.println("hello, java 21")
hello, java 21

jshell> var name = "ganesh"
name ==> "ganesh"

jshell> "hello, " + name
$3 ==> "hello, ganesh"

jshell> /exit

Three details to notice in that transcript:

- `var name = "ganesh"` declares a local variable. The compiler infers the type as `String` — no need to write it out.
- A bare expression like `"hello, " + name` is evaluated, and the result is bound to an auto-generated name like `$3` you can refer to in the next prompt.
- `/exit` quits. (`/help` shows all `jshell` commands — they all start with `/`.)

## Hello, jbang

For anything longer than two lines, put it in a file and run it with `jbang`. Create `hello.java`:

In [ ]:
///usr/bin/env jbang "$0" "$@" ; exit $?
//JAVA 21

public class hello {
    public static void main(String[] args) {
        System.out.println("hello from jbang");
    }
}

Run it from the same directory:

In [ ]:
%%bash
jbang hello.java
# hello from jbang

Three new shapes appeared:

- The shebang line at the top lets you run `./hello.java` directly once you `chmod +x` it. `jbang` is the interpreter.
- `//JAVA 21` is a `jbang` directive that pins the language version for this file.
- `public static void main(String[] args)` is Java's entry point signature. Every standalone Java program starts in a method with this exact shape. Modern Java has a preview *implicit main* that drops the boilerplate, but the classic form is still what you will read everywhere.

## Hello, Maven

For a real project, **Maven** gives you a directory layout, dependency management, and a test runner. You will meet it properly in notebook 07. For now, the one-command scaffold is enough:

In [ ]:
%%bash
mvn archetype:generate \
  -DgroupId=com.example \
  -DartifactId=myapp \
  -DarchetypeArtifactId=maven-archetype-quickstart \
  -DinteractiveMode=false

That creates a directory with the standard Maven layout:

```text
  myapp/
  +-- pom.xml                 <- project config: name, version, java version, deps
  +-- src/
       +-- main/java/         <- production code goes here
       +-- test/java/         <- tests go here
```

Run `mvn compile exec:java -Dexec.mainClass=com.example.App` to run the generated `App.java`.

## Values, types and the type system

Every value in Java has a **type** the compiler knows at compile time. Types come in two flavours:

- **Primitive types** — built into the language, stored directly: `int`, `long`, `double`, `float`, `boolean`, `char`, `byte`, `short`.
- **Reference types** — every class is a reference type. A variable of a reference type holds a **reference** (essentially a pointer) to an object on the heap. `String`, `List<Integer>`, arrays, and any class you define are reference types.

The distinction matters because primitives live on the stack with their value embedded, while reference-typed values live as references that point to a heap-allocated object. We will return to this in notebook 07 when we talk about JVM memory.

In [ ]:
// primitives — value stored directly
int count = 42;
long timestamp = 1_700_000_000_000L;   // underscores are visual separators
double pi = 3.14159;
boolean ready = true;
char letter = 'a';

// reference types — variable holds a reference to a heap object
String name = "ganesh";
int[] scores = {90, 85, 78};
java.util.List<Integer> list = java.util.List.of(1, 2, 3);

Every primitive has a **boxed** reference-type partner: `int` ↔ `Integer`, `long` ↔ `Long`, `boolean` ↔ `Boolean`, and so on. The compiler **autoboxes** between them when needed — `Integer x = 42` works because `42` is silently wrapped in `Integer.valueOf(42)`. Autoboxing is convenient but allocates an object, so in hot loops you prefer primitives.

## `var` and local type inference

Since Java 10, you can omit the type on a local variable declaration and let the compiler infer it from the right-hand side. This is purely a syntactic convenience — Java is still statically typed, and the inferred type is fixed at compile time.

In [ ]:
var count = 42;                      // inferred: int
var name = "ganesh";                 // inferred: String
var scores = new int[]{90, 85, 78};  // inferred: int[]
var users = new java.util.ArrayList<String>();  // inferred: ArrayList<String>

// var is only allowed on local variables with an initializer.
// var x;          // ERROR — no initializer
// var y = null;   // ERROR — null has no inferable type
// public var foo = 1;  // ERROR — not allowed on fields

Use `var` when the type is obvious from the right-hand side and adds nothing to readability. Spell the type out when the right-hand side is a method call whose return type is non-obvious. The compiler does not care; future-you reading the file does.

## Expressions vs statements

A core distinction in Java's grammar:

- An **expression** produces a value. `1 + 2` is an expression; so is `name.length()`; so is the ternary `cond ? a : b`.
- A **statement** performs an action and does not produce a value. `int x = 5;` is a statement; so is `if (cond) { ... }`; so is `while (...) { ... }`.

This is the opposite of Scala, where `if`, `while`, and `for` are expressions. In Java they are statements — they cannot appear on the right-hand side of an assignment. The newer **switch expression** (Java 14+) is the one place Java added a real expression form for control flow.

In [ ]:
// expressions — produce a value
int sum = 1 + 2;                  // 1 + 2 is the expression
int len = "hello".length();       // method call is an expression
String label = sum > 0 ? "pos" : "neg";  // ternary is an expression

// statements — do not produce a value
int x = 5;                        // declaration statement
x = x + 1;                        // assignment statement (the assignment is technically an expression, the ; makes it a statement)
if (x > 0) { System.out.println("positive"); }  // if-statement

// switch expression (Java 14+) — the modern form that DOES produce a value
String day = switch (x % 7) {
    case 0 -> "sun";
    case 1 -> "mon";
    default -> "other";
};

## Operators

Java's operators look like C, Python, and JavaScript. The arithmetic and comparison ones are unsurprising; a few have Java-specific behaviour worth flagging.

- **Arithmetic**: `+`, `-`, `*`, `/`, `%`. Integer division truncates toward zero — `7 / 2` is `3`, not `3.5`.
- **Comparison**: `<`, `<=`, `>`, `>=`, `==`, `!=`. For primitives, `==` compares values. For reference types, `==` compares *references* — use `.equals()` to compare object content.
- **Logical**: `&&`, `||`, `!`. Both `&&` and `||` short-circuit.
- **Bitwise**: `&`, `|`, `^`, `~`, `<<`, `>>`, `>>>`. `>>>` is the unsigned right shift, a Java-specific operator.
- **Assignment shortcuts**: `+=`, `-=`, `*=`, etc., and the increment/decrement `++` and `--`.

In [ ]:
int a = 7, b = 2;

a / b;     // 3   — integer division truncates
a % b;     // 1   — modulo
(double) a / b;  // 3.5 — cast one side to force floating-point division

// reference equality vs value equality — the classic Java gotcha
String s1 = "hello";
String s2 = "hello";
String s3 = new String("hello");
s1 == s2;        // true  — string literals share the same intern pool
s1 == s3;        // false — `new String` always allocates a fresh object
s1.equals(s3);   // true  — content comparison

The `==` versus `.equals()` distinction trips every new Java developer at least once. Burn it in now: **`==` on objects compares identity, `.equals()` compares content.** The only safe use of `==` on reference types is to check for `null`.

## Control flow statements

Java's control flow is the C family with a few modern additions.

- **`if` / `else if` / `else`** — the conditional statement.
- **`while`** and **`do-while`** — pre- and post-test loops.
- **`for`** — classic three-clause loop, plus the **enhanced for** (`for (var x : collection)`) for iteration.
- **`switch`** — both classic statement form and modern expression form.
- **`break`** and **`continue`** — loop control.

In [ ]:
// if / else if / else
int score = 87;
if (score >= 90) {
    System.out.println("A");
} else if (score >= 80) {
    System.out.println("B");
} else {
    System.out.println("C or below");
}

// classic for
for (int i = 0; i < 3; i++) {
    System.out.println(i);
}

// enhanced for — iterate over any Iterable or array
var names = java.util.List.of("alice", "bob", "carol");
for (var name : names) {
    System.out.println(name);
}

// while
int n = 5;
while (n > 0) {
    System.out.println(n);
    n--;
}

// switch expression (Java 14+) with pattern arrows
String label = switch (score / 10) {
    case 10, 9 -> "A";
    case 8     -> "B";
    case 7     -> "C";
    default    -> "F";
};

## Methods

A **method** is a named, reusable block of code that lives inside a class. Every Java method declares its return type, its name, and its parameter list:

```text
   modifiers   return-type   name   ( parameters )   { body }
```

In [ ]:
public class MathUtil {

    // a static method — called as MathUtil.square(5)
    public static int square(int x) {
        return x * x;
    }

    // overloading — same name, different parameter types
    public static double square(double x) {
        return x * x;
    }

    // a method that returns nothing uses `void`
    public static void greet(String name) {
        System.out.println("hello, " + name);
    }
}

// calling
MathUtil.square(5);      // 25
MathUtil.square(2.5);    // 6.25
MathUtil.greet("ganesh");

Two language features worth flagging:

- **Overloading** — multiple methods can share a name as long as their parameter lists differ. The compiler picks the right one based on the argument types at the call site.
- **`void`** — Java's "no return value" marker. Unlike Scala's `Unit`, `void` is not a real type; it is a syntactic marker. You cannot declare a `void` variable.

## Strings and text blocks

`String` is the most-used class in Java. Strings are **immutable** — every operation that looks like it modifies a string actually returns a new one.

In [ ]:
String greeting = "hello";
String shouted = greeting.toUpperCase();    // returns a new String
System.out.println(greeting);  // still "hello"
System.out.println(shouted);   // "HELLO"

// concatenation
String name = "ganesh";
String hi = "hello, " + name;               // works, but each + allocates
String hi2 = String.format("hello, %s", name);   // printf-style
String hi3 = "hello, %s".formatted(name);        // Java 15+ instance method

For multi-line strings, Java 15 added **text blocks** — triple-quoted strings that preserve newlines and indentation rules. They are the cleanest way to embed JSON, SQL, or HTML in code.

In [ ]:
String json = """
        {
          "name": "ganesh",
          "role": "engineer"
        }
        """;

String sql = """
        SELECT id, name
          FROM users
         WHERE active = true
        """;

System.out.println(json);

The compiler strips a common leading whitespace prefix from every line of a text block, so you can indent the literal to match the surrounding code without that indentation appearing in the resulting string. The trailing `"""` on its own line sets the baseline.

## Comments

Same conventions as C and Scala, plus Javadoc:

In [ ]:
// single-line comment

/*
   multi-line block comment
*/

/**
 * Javadoc comment — used to document public APIs.
 * Tools like `javadoc` turn these into HTML reference docs.
 *
 * @param name the person to greet
 * @return the greeting string
 */
public static String greet(String name) {
    return "hello, " + name;
}

## What's next

You now have Java 21 installed, you can poke at it in `jshell`, and you can run a one-file script with `jbang` or a full Maven project. You also have the vocabulary in place: values, primitive vs reference types, expressions vs statements, operators, `var`, control flow, methods, strings, and text blocks.

Notebook 02 starts object-oriented Java in earnest: classes, inheritance, interfaces, abstract classes, enums, and the modern additions that change how you model data — **records**, **sealed types**, and **pattern matching for `switch`**. These are the shapes Spring will lean on later, so it is worth getting fluent with them now.